# Zeit- und Frequenzbereich

Analyse eines Audiosignals im Zeit- und Frequenzbereich
mit Schmalband- und Terzbandspektrum sowie A-Bewertung.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoeverc/laermarmesysteme_medien/blob/main/Python/VL2/ZeitUndFrequenzbereich.ipynb)

In [ ]:
import os
import urllib.request

GITHUB_RAW=(
  'https://raw.githubusercontent.com'
  '/hoeverc/laermarmesysteme_medien/main')

def download(url, path):
  """Download file if not already present."""
  os.makedirs(
    os.path.dirname(path) or '.',
    exist_ok=True)
  if not os.path.exists(path):
    urllib.request.urlretrieve(url, path)
    print(f'Heruntergeladen: {path}')

# Audio-Datei
download(
  f'{GITHUB_RAW}/VL2/Medien/musik.wav',
  'Medien/musik.wav')

# Hilfsfunktionen
for mod in [
    'aweight.py',
    'aweight_narrow.py',
    'get_third_octave_band_spectrum.py']:
  download(f'{GITHUB_RAW}/VL2/{mod}', mod)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from IPython.display import Audio, display
import ipywidgets as widgets

from aweight import aweight
from aweight_narrow import aweight_narrow
from get_third_octave_band_spectrum import (
  get_third_octave_band_spectrum)

# ===========================================================
# Einstellungen
# ===========================================================
freqlims=[13, 18000]
t_freq_ana=2      # Analysesegment-Laenge, s
t_offset=10.5     # Start des Analysesegments, s
fontsize=12
linewidth=2

# ===========================================================
# Audio einlesen
# ===========================================================
sr, data=wavfile.read('Medien/musik.wav')
if data.ndim>1:
  data=data[:, 0]
if np.issubdtype(data.dtype, np.integer):
  x=data.astype(np.float64)/np.iinfo(data.dtype).max
else:
  x=data.astype(np.float64)
fs=sr

# ===========================================================
# Signalverarbeitung
# ===========================================================
# Analysesegment
ind_start=round(t_offset*fs)
n_ana=t_freq_ana*fs
ind_end=ind_start+n_ana
n_tot=len(x)

# RMS-Normierung auf 100 dB SPL
x_rms=np.sqrt(np.mean(x**2))
target_rms=10**(100/20)*2e-5
x_norm=x/x_rms*target_rms
x_ana=x_norm[ind_start:ind_end]

# Zeitvektoren
T_tot=n_tot/fs
t_tot=np.arange(n_tot)/fs

# Frequenzdaten
fmax=fs/2
f=np.linspace(0, fmax, n_ana//2+1)

# FFT: einseitiges Amplitudenspektrum
X_full=np.fft.fft(x_ana)/n_ana
X=2*X_full[:n_ana//2+1].copy()
X[0]/=2
X[-1]/=2

# Schalldruckpegel (Schmalband)
Lp=20*np.log10(np.abs(X)/np.sqrt(2)/2e-5)
LpA=aweight_narrow(f, Lp)
Lp_tot=10*np.log10(np.sum(10**(Lp/10)))
LpA_tot=10*np.log10(np.sum(10**(LpA/10)))

# Terzbandspektrum
f_mask=(f>=freqlims[0]) & (f<=freqlims[1])
f_third, Lp_third=get_third_octave_band_spectrum(
  f[f_mask], Lp[f_mask])
LpA_third=aweight(f_third, Lp_third)

## Audio abspielen

In [ ]:
display(Audio(data=x, rate=fs))

## Analyse

Nutzen Sie die Schalter **Terzen** und **A-Bewertung**,
um Terzband-Darstellung und A-Bewertung ein-/auszublenden.

In [ ]:
def plot_all(show_terzen=False,
             show_a_bewertung=False):
  """Plot des Zeit-/Frequenzbereichs."""
  fig, axes=plt.subplots(
    3, 1, figsize=(12, 12))

  # --- Zeitsignal ---
  ax=axes[0]
  ax.plot(t_tot, x_norm, linewidth=linewidth)
  ax.axvspan(
    t_tot[ind_start],
    t_tot[min(ind_end-1, n_tot-1)],
    alpha=0.3, color='gray')
  ax.set_xlabel(
    r'$t$ in s', fontsize=fontsize)
  ax.set_ylabel(
    r'$p(t)$ in Pa', fontsize=fontsize)
  ax.set_xlim(0, T_tot)
  ax.set_title('Zeitsignal')
  ax.tick_params(labelsize=fontsize)

  # --- linearer Schalldruck ---
  ax=axes[1]
  ax.semilogx(
    f, np.abs(X), linewidth=linewidth)
  ax.set_xlabel(
    r'$f$ in Hz', fontsize=fontsize)
  ax.set_ylabel(
    r'$p(f)$ in Pa', fontsize=fontsize)
  ax.set_xlim(*freqlims)
  ax.set_title(
    'linearer Schalldruck als '
    'Schmalbandspektrum')
  ax.tick_params(labelsize=fontsize)

  # --- Schalldruckpegel ---
  ax=axes[2]
  ax.plot(f, Lp, 'b', linewidth=linewidth,
    label='Schmalband')

  if show_a_bewertung:
    ax.plot(f, LpA, 'r', linewidth=1,
      label='Schmalband A-bewertet')

  if show_terzen:
    for ff in range(len(f_third)):
      lo=f_third[ff]/2**(1/6)
      hi=f_third[ff]*2**(1/6)
      lbl=('Terzband'
        if ff==0 else '_nolegend_')
      ax.plot([lo, hi],
        [Lp_third[ff], Lp_third[ff]],
        'c', linewidth=linewidth,
        label=lbl)
      if show_a_bewertung:
        lbl_a=('Terzband A-bewertet'
          if ff==0 else '_nolegend_')
        ax.plot([lo, hi],
          [LpA_third[ff], LpA_third[ff]],
          'm', linewidth=linewidth,
          label=lbl_a)

    # Total level
    tot_lo=(
      f_third[-1]*2**(1/3)/2**(1/6))
    tot_hi=(
      f_third[-1]*2**(1/3)*2**(1/6))
    ax.plot([tot_lo, tot_hi],
      [Lp_tot, Lp_tot], 'c',
      linewidth=1.5*linewidth)
    if show_a_bewertung:
      ax.plot([tot_lo, tot_hi],
        [LpA_tot, LpA_tot], 'm',
        linewidth=1.5*linewidth)

    ax.set_title(
      'Schalldruckpegel als '
      'Schmal- und Terzbandspektrum')
    upper_lim=(
      f_third[-1]*2**(1/3)*2**(1/6))
    ax.set_xlim(freqlims[0], upper_lim)
    ax.set_ylim(0, Lp_tot+5)

    # Xtick-Labels mit 'Total'
    xtv=np.append(f_third[::2], 25000)
    xtl=([str(int(v))
      for v in f_third[::2]]+['Total'])
    ax.set_xticks(xtv)
    ax.set_xticklabels(xtl)
  else:
    ax.set_title(
      'Schalldruckpegel als '
      'Schmalbandspektrum')
    ax.set_xlim(*freqlims)
    ax.set_ylim(
      0, np.max(Lp_third)+5)

  ax.set_xscale('log')
  ax.set_xlabel(
    r'$f$ in Hz', fontsize=fontsize)
  if show_a_bewertung:
    ax.set_ylabel(
      r'$dB$ ($dBA$) re 2e-5 Pa',
      fontsize=fontsize)
  else:
    ax.set_ylabel(
      r'$dB$ re 2e-5 Pa',
      fontsize=fontsize)
  ax.legend(
    loc='lower right', fontsize=fontsize)
  ax.tick_params(labelsize=fontsize)

  plt.tight_layout()
  plt.show()

widgets.interact(
  plot_all,
  show_terzen=widgets.ToggleButton(
    value=False, description='Terzen'),
  show_a_bewertung=widgets.ToggleButton(
    value=False,
    description='A-Bewertung'))